# Model fitting (development CV)

Pipeline: `EDA_notebook.Rmd` → `eda_training.csv` → drop NA-`asthma` (same as `validation_baseline.R`) → 5-fold CV with existing `fold_id`.

Models: (1) Linear regression (2) Lasso (`LassoCV` within each training fold) (3) Random forest.

**Feature sets:** every algorithm is fit on **two** feature sets (defined in Section 0b, with rationale):
- **A_base**: all 13 raw predictors + one-hot categoricals (21 columns).
- **B_poly**: A_base + centered squared (quadratic) terms for predictors whose EDA shows curvature.

This gives 3 models x 2 feature sets = 6 fits, each evaluated with the same 5-fold CV.

Metrics: mean of fold RMSEs **and** pooled OOF RMSE. 

## 0. Load data 

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RF_PARAMS = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def resolve_path(candidates):
    for p in candidates:
        path = Path(p)
        if path.exists():
            return path
    raise FileNotFoundError(
        "None of these paths exist:\n  - " + "\n  - ".join(map(str, candidates))
    )

In [12]:
csv_path = resolve_path(
    [
        Path("../data/clean_data/eda_training.csv"),
        Path("data/clean_data/eda_training.csv"),
    ]
)

raw = pd.read_csv(csv_path)
print(f"Loaded {csv_path.resolve()}")
print(f"Raw shape: {raw.shape}")

# Same NA-asthma drop as validation_baseline.R (before any CV)
asthma = pd.to_numeric(raw["asthma"], errors="coerce")
n_before = len(raw)
df = raw.loc[asthma.notna()].copy()
df["asthma"] = asthma.loc[asthma.notna()].astype(float)
df["fold_id"] = df["fold_id"].astype(int)

print(f"Dropped NA asthma: {n_before - len(df)} rows")
print(f"Eligible shape: {df.shape}")
print("Fold counts:\n", df["fold_id"].value_counts().sort_index())

assert not df["asthma"].isna().any()
assert set(df["fold_id"]) == {1, 2, 3, 4, 5}

ID_COLS = ["census_tract", "california_county", "fold_id"]
TARGET = "asthma"
CAT_COLS = ["region", "haz_waste_any", "solid_waste_any"]
CAT_COLS = [c for c in CAT_COLS if c in df.columns]

feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET]]
num_cols = [c for c in feature_cols if c not in CAT_COLS]

# One-hot categoricals once (closed category set); scaling stays fold-wise for linear/lasso
X_all = pd.get_dummies(df[feature_cols], columns=CAT_COLS, drop_first=False)
y_all = df[TARGET].to_numpy(dtype=float)
fold_id = df["fold_id"].to_numpy(dtype=int)
feature_names = list(X_all.columns)
X_mat = X_all.to_numpy(dtype=float)

print(f"Feature matrix: {X_mat.shape} (numeric + one-hot)")
print("Features:", feature_names)

# Containers filled by model sections; evaluation section consumes these
oof_preds = {}
fold_rmses = {}
model_notes = {}

Loaded C:\Users\antro\OneDrive\Documents\Stanford\ClassCodes\MSE226\Proj1\data\clean_data\eda_training.csv
Raw shape: (6428, 19)
Dropped NA asthma: 0 rows
Eligible shape: (6428, 19)
Fold counts:
 fold_id
1    1286
2    1286
3    1286
4    1285
5    1285
Name: count, dtype: int64
Feature matrix: (6428, 21) (numeric + one-hot)
Features: ['ozone', 'pm2_5', 'diesel_pm', 'drinking_water', 'pesticides', 'tox_release', 'traffic', 'cleanup_sites', 'groundwater_threats', 'imp_water_bodies', 'linguistic_isolation', 'poverty', 'unemployment', 'region_Bay Area', 'region_Other', 'region_San Joaquin Valley', 'region_SoCal', 'haz_waste_any_no', 'haz_waste_any_yes', 'solid_waste_any_no', 'solid_waste_any_yes']


## 0b. Feature sets

Each algorithm below is fit on both sets so differences in CV error can be attributed to the features rather than the algorithm.

**Set A: `A_base` (raw predictors).** All 13 numeric predictors plus one-hot `region`, `haz_waste_any`, `solid_waste_any`. This is the reference set: it assumes each predictor affects asthma ED visit rate *linearly* (for linear/Lasso) and lets the random forest discover any structure on its own.

**Set B: `B_poly` (raw + quadratic terms).** Adds the square of the mean-centered value, `(x - mean_train)^2`, for five predictors:

| Predictor | Why a quadratic term (EDA: single-predictor R² linear → quadratic) |
|---|---|
| `poverty` | Strongest predictor (R² 0.229 → 0.235). Spearman (0.556) exceeds Pearson (0.478), so the relationship is monotone but not perfectly linear. The tract-level rate may rise faster at high poverty. |
| `unemployment` | Second strongest (0.171 → 0.186). Right-skewed (skew ≈ 2, max 100), so a few high-unemployment tracts may behave differently from the rest. |
| `linguistic_isolation` | Quadratic gain is the largest among socioeconomic variables (0.029 → 0.053). Spearman (0.295) far exceeds Pearson (0.17), which signals curvature or saturation. |
| `ozone` | Largest relative gain (0.004 → 0.032). A near-zero linear fit with a clear quadratic fit suggests a non-monotone (U-shaped) relationship, which a linear term cannot capture. |
| `pm2_5` | 0.009 → 0.025. Same logic as ozone: the effect may flatten or reverse at high concentrations. |

**Why this might affect prediction.**
- *Linear / Lasso:* these models can only fit straight-line effects, so squared terms let them represent curvature. This should lower bias. The cost is extra correlated columns, which raises variance for OLS. Lasso can shrink unhelpful squared terms to zero.
- *Random forest:* trees already capture nonlinearities through splits. Squared terms are mostly redundant, so we expect little or no change. This acts as a check that any gain for linear/Lasso comes from nonlinearity and not from extra capacity.

**Leakage control.** Centering means are computed on the **training fold only** and applied to the validation fold. Centering also reduces the correlation between `x` and `x^2`.

Predictors **not** squared (`pesticides`, `tox_release`, `traffic`, `groundwater_threats`, `cleanup_sites`, `drinking_water`, `imp_water_bodies`, `diesel_pm`) either have near-zero association with the outcome, or are extremely right-skewed (skew 4 to 18), so their squares would be dominated by a handful of outlier tracts and add variance without signal.

In [14]:
POLY_COLS = ["poverty", "unemployment", "linguistic_isolation", "ozone", "pm2_5"]
missing = [c for c in POLY_COLS if c not in X_all.columns]
assert not missing, f"POLY_COLS missing from data: {missing}"


def build_A_base(X_train, X_val):
    """Set A: raw predictors + one-hot categoricals."""
    return X_train.to_numpy(dtype=float), X_val.to_numpy(dtype=float)


def build_B_poly(X_train, X_val):
    """Set B: Set A + squared mean-centered POLY_COLS (means from the training fold only)."""
    mu = X_train[POLY_COLS].mean()

    def augment(X):
        sq = (X[POLY_COLS] - mu) ** 2
        sq.columns = [f"{c}_sq" for c in POLY_COLS]
        return pd.concat([X, sq], axis=1)

    return augment(X_train).to_numpy(dtype=float), augment(X_val).to_numpy(dtype=float)


FEATURE_SETS = {"A_base": build_A_base, "B_poly": build_B_poly}

for name, builder in FEATURE_SETS.items():
    Xt, _ = builder(X_all, X_all.iloc[:5])
    print(f"{name}: {Xt.shape[1]} columns")

A_base: 21 columns
B_poly: 26 columns


In [15]:
def run_fold_cv(fit_predict_fn, name, feature_set):
    """Train on each fold's complement; score held-out fold.

    The feature set builder runs inside each fold so any statistics it uses
    (e.g. centering means) come from the training fold only.
    """
    builder = FEATURE_SETS[feature_set]
    oof = np.full(len(y_all), np.nan)
    per_fold = []

    for k in range(1, 6):
        train = fold_id != k
        val = fold_id == k

        X_tr, X_va = builder(X_all.loc[train], X_all.loc[val])
        y_hat_val = fit_predict_fn(X_tr, y_all[train], X_va)
        oof[val] = y_hat_val
        fold_rmse = rmse(y_all[val], y_hat_val)
        per_fold.append({"fold": k, "validation_n": int(val.sum()), "validation_rmse": fold_rmse})
        print(f"{name} | fold {k}: RMSE={fold_rmse:.4f} (n={int(val.sum())})")

    assert not np.isnan(oof).any()
    fold_table = pd.DataFrame(per_fold)
    print(
        f"{name} | mean fold RMSE={fold_table['validation_rmse'].mean():.4f} | "
        f"pooled OOF RMSE={rmse(y_all, oof):.4f}"
    )
    return oof, fold_table

## 1. Linear regression

Standardize `X` on the **training fold only**, then OLS. Fit on both `A_base` and `B_poly`.

OLS is the model most likely to benefit from `B_poly`, because it can only represent straight-line effects. It is also the most exposed to the extra variance from added columns.

In [16]:
def fit_predict_linear(X_train, y_train, X_val):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_va = scaler.transform(X_val)
    model = LinearRegression()
    model.fit(X_tr, y_train)
    return model.predict(X_va)


for fs in FEATURE_SETS:
    key = f"linear|{fs}"
    oof_preds[key], fold_rmses[key] = run_fold_cv(
        fit_predict_linear, f"Linear [{fs}]", fs
    )
    model_notes[key] = "OLS + StandardScaler fit on each training fold"

Linear [A_base] | fold 1: RMSE=24.8693 (n=1286)
Linear [A_base] | fold 2: RMSE=24.7343 (n=1286)
Linear [A_base] | fold 3: RMSE=24.5900 (n=1286)
Linear [A_base] | fold 4: RMSE=24.6304 (n=1285)
Linear [A_base] | fold 5: RMSE=24.1073 (n=1285)
Linear [A_base] | mean fold RMSE=24.5863 | pooled OOF RMSE=24.5877
Linear [B_poly] | fold 1: RMSE=24.6130 (n=1286)
Linear [B_poly] | fold 2: RMSE=24.5104 (n=1286)
Linear [B_poly] | fold 3: RMSE=24.3427 (n=1286)
Linear [B_poly] | fold 4: RMSE=24.3508 (n=1285)
Linear [B_poly] | fold 5: RMSE=23.9206 (n=1285)
Linear [B_poly] | mean fold RMSE=24.3475 | pooled OOF RMSE=24.3487


## 2. Lasso regression

Standardize `X` on the training fold, then `LassoCV` (inner CV on that training fold only) to pick `alpha`. Fit on both `A_base` and `B_poly`.

Lasso's L1 penalty can zero out squared terms that do not help, so it should handle the larger `B_poly` set more gracefully than OLS.

In [17]:
def fit_predict_lasso(X_train, y_train, X_val):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_va = scaler.transform(X_val)
    model = LassoCV(
        cv=5,
        random_state=RANDOM_STATE,
        max_iter=10000,
        n_jobs=-1,
    )
    model.fit(X_tr, y_train)
    return model.predict(X_va)


for fs in FEATURE_SETS:
    key = f"lasso|{fs}"
    oof_preds[key], fold_rmses[key] = run_fold_cv(
        fit_predict_lasso, f"Lasso [{fs}]", fs
    )
    model_notes[key] = "LassoCV(cv=5) + StandardScaler fit on each training fold"

Lasso [A_base] | fold 1: RMSE=24.9369 (n=1286)
Lasso [A_base] | fold 2: RMSE=24.7519 (n=1286)
Lasso [A_base] | fold 3: RMSE=24.5830 (n=1286)
Lasso [A_base] | fold 4: RMSE=24.6483 (n=1285)
Lasso [A_base] | fold 5: RMSE=24.0773 (n=1285)
Lasso [A_base] | mean fold RMSE=24.5995 | pooled OOF RMSE=24.6012
Lasso [B_poly] | fold 1: RMSE=24.6922 (n=1286)
Lasso [B_poly] | fold 2: RMSE=24.4388 (n=1286)
Lasso [B_poly] | fold 3: RMSE=24.3285 (n=1286)
Lasso [B_poly] | fold 4: RMSE=24.3719 (n=1285)
Lasso [B_poly] | fold 5: RMSE=23.8860 (n=1285)
Lasso [B_poly] | mean fold RMSE=24.3435 | pooled OOF RMSE=24.3450


## 3. Random forest

Fixed hyperparameters; **no** feature scaling. Fit on both `A_base` and `B_poly`.

Trees split on thresholds, so they already capture curvature in the raw predictors. We expect `B_poly` to give little or no improvement here, which helps confirm that any gain for linear/Lasso is due to nonlinearity.

In [18]:
def fit_predict_rf(X_train, y_train, X_val):
    model = RandomForestRegressor(**RF_PARAMS)
    model.fit(X_train, y_train)
    return model.predict(X_val)


print("RF params:", RF_PARAMS)
for fs in FEATURE_SETS:
    key = f"random_forest|{fs}"
    oof_preds[key], fold_rmses[key] = run_fold_cv(
        fit_predict_rf, f"RandomForest [{fs}]", fs
    )
    model_notes[key] = f"RandomForestRegressor{RF_PARAMS}"

RF params: {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 5, 'random_state': 42, 'n_jobs': -1}
RandomForest [A_base] | fold 1: RMSE=21.7697 (n=1286)
RandomForest [A_base] | fold 2: RMSE=22.0110 (n=1286)
RandomForest [A_base] | fold 3: RMSE=20.9620 (n=1286)
RandomForest [A_base] | fold 4: RMSE=20.9693 (n=1285)
RandomForest [A_base] | fold 5: RMSE=20.2441 (n=1285)
RandomForest [A_base] | mean fold RMSE=21.1912 | pooled OOF RMSE=21.2008
RandomForest [B_poly] | fold 1: RMSE=21.6815 (n=1286)
RandomForest [B_poly] | fold 2: RMSE=21.8977 (n=1286)
RandomForest [B_poly] | fold 3: RMSE=20.8947 (n=1286)
RandomForest [B_poly] | fold 4: RMSE=21.0782 (n=1285)
RandomForest [B_poly] | fold 5: RMSE=20.2379 (n=1285)
RandomForest [B_poly] | mean fold RMSE=21.1580 | pooled OOF RMSE=21.1664


## 4. Comparison across algorithms and feature sets

Winner is chosen by **pooled OOF RMSE**. `delta_vs_A` is the change from using `B_poly` instead of `A_base` for the same algorithm (negative means the engineered features helped).

In [19]:
rows = []
for key, oof in oof_preds.items():
    algo, fs = key.split("|")
    rows.append(
        {
            "algorithm": algo,
            "feature_set": fs,
            "mean_fold_rmse": fold_rmses[key]["validation_rmse"].mean(),
            "pooled_oof_rmse": rmse(y_all, oof),
        }
    )

summary = pd.DataFrame(rows)
base_rmse = summary[summary["feature_set"] == "A_base"].set_index("algorithm")["pooled_oof_rmse"]
summary["delta_vs_A"] = summary["pooled_oof_rmse"] - summary["algorithm"].map(base_rmse)
summary = summary.sort_values("pooled_oof_rmse").reset_index(drop=True)

# Reference: training-mean baseline, pooled over folds (same aggregation)
mean_oof = np.empty_like(y_all)
for k in range(1, 6):
    mean_oof[fold_id == k] = y_all[fold_id != k].mean()
print(f"Training-mean baseline pooled OOF RMSE: {rmse(y_all, mean_oof):.4f}\n")

print(summary.round(4).to_string(index=False))
best = summary.iloc[0]
print(f"\nBest: {best['algorithm']} on {best['feature_set']} (pooled OOF RMSE={best['pooled_oof_rmse']:.4f})")

Training-mean baseline pooled OOF RMSE: 30.4183

    algorithm feature_set  mean_fold_rmse  pooled_oof_rmse  delta_vs_A
random_forest      B_poly         21.1580          21.1664     -0.0344
random_forest      A_base         21.1912          21.2008      0.0000
        lasso      B_poly         24.3435          24.3450     -0.2562
       linear      B_poly         24.3475          24.3487     -0.2390
       linear      A_base         24.5863          24.5877      0.0000
        lasso      A_base         24.5995          24.6012      0.0000

Best: random_forest on B_poly (pooled OOF RMSE=21.1664)
